# Episode 1 — micrograd

Follow-along for Karpathy's [building micrograd](https://www.youtube.com/watch?v=VMj-3S1tku0) (~2h25m).

**Rule:** type the code, don't paste it. Pause the video when a section ends and make sure you understand before moving on.

Section headers below match the video chapter markers.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Derivative of a simple function with one input

Numerical derivative: bump `x` by a tiny `h` and see how much `f(x)` changes.

In [ ]:
# def f(x): ...
# numerical derivative via finite differences


## 2. Derivative of a function with multiple inputs

Same idea, but for `d = a*b + c`. One partial derivative per input.

## 3. Starting the core `Value` object

Wrap a scalar so we can track which `Value`s produced it (`_children`) and what op (`_op`).

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        # TODO: _prev, _op, label, grad
    
    def __repr__(self):
        return f"Value(data={self.data})"
    
    # def __add__(self, other): ...
    # def __mul__(self, other): ...

## 4. Visualizing the expression graph

Use `graphviz` to draw the computation graph — makes backprop easier to reason about.

In [ ]:
from graphviz import Digraph

def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'})
    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        dot.node(name=uid, label=f"{{ {n.label} | data {n.data:.4f} | grad {n.grad:.4f} }}", shape='record')
        if n._op:
            dot.node(name=uid + n._op, label=n._op)
            dot.edge(uid + n._op, uid)
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    return dot

## 5. Manual backprop — example #1

Walk the graph by hand: for `L = d * f`, what is `dL/dL`? `dL/dd`? `dL/df`? Keep going.

## 6. Implementing `.backward()` per-op

Each op (`+`, `*`, `tanh`) attaches a closure `_backward` that knows how to push gradients to its children via the chain rule.

## 7. Topological sort + full `.backward()`

Build a topo order so we visit each node only after its children have their gradients.

In [ ]:
# def backward(self):
#     topo = []
#     visited = set()
#     def build_topo(v): ...
#     self.grad = 1.0
#     for node in reversed(topo): node._backward()

## 8. The accumulation bug

Karpathy's classic gotcha: if a Value is used twice, gradients must **accumulate** (`+=`), not overwrite (`=`).

## 9. `tanh` from scratch + breaking it down

Show that `tanh` can be one op or decomposed into `exp`, `+`, `-`, `/` — the gradients come out the same.

## 10. Sanity check vs. PyTorch

Build the same expression with `torch.Tensor(..., requires_grad=True)`, call `.backward()`, and check our grads match.

In [ ]:
import torch

## 11. Building a `Neuron`

`Neuron(nin)` holds `nin` weights + a bias, all `Value`s. Forward = `tanh(sum(w_i * x_i) + b)`.

In [ ]:
import random

class Neuron:
    def __init__(self, nin):
        pass  # TODO
    def __call__(self, x):
        pass  # TODO

## 12. Building a `Layer`

`Layer(nin, nout)` = list of `nout` neurons, each takes `nin` inputs. Calling it returns a list of activations.

## 13. Building an `MLP`

`MLP(nin, [n1, n2, ..., nout])` = stack of layers. End-to-end forward pass.

## 14. Loss + gradient descent

Mean squared error against target outputs. Then a manual training loop:

1. forward → loss
2. zero grads
3. `loss.backward()`
4. nudge every parameter: `p.data -= lr * p.grad`
5. repeat

In [ ]:
# xs = [[2.0, 3.0, -1.0], [3.0, -1.0, 0.5], [0.5, 1.0, 1.0], [1.0, 1.0, -1.0]]
# ys = [1.0, -1.0, -1.0, 1.0]
# n = MLP(3, [4, 4, 1])
# for k in range(200):
#     ...
#     print(k, loss.data)

## 15. Recap

- Autograd = computation graph + chain rule walked in reverse topological order.
- A neural net is just a big expression. PyTorch does the same thing on tensors, with CUDA, and more ops — but the core idea is what we built here.
- Once this clicks, Episode 2 (makemore) uses it to train a real language model.